# SCD2 History — Baseline & Preflight

Notebook này chỉ đọc dữ liệu. Mục tiêu là ghi lại trạng thái Bronze, kiểm tra schema và xác nhận baseline ngày `2025-12-31` trước khi tạo thay đổi day-2.

In [1]:
BASE_COB_DT = "2025-12-31"
SCD2_DIMS = {
    "dim_branch": ("branch_code", "branch_sk"),
    "dim_product": ("product_code", "product_sk"),
    "dim_customer": ("customer_id", "customer_sk"),
    "dim_account": ("account_id", "account_sk"),
}

try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

print(f"Spark {spark.version}")
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

Spark 3.5.0


+---------+------+
|catalog  |schema|
+---------+------+
|lakehouse|      |
+---------+------+



## 1. Bronze inventory

Counts này là checkpoint để đối chiếu việc migration Silver/Gold không làm thay đổi Bronze.

In [2]:
BRONZE_TABLES = [
    "core_branch", "core_product", "core_customer", "core_account",
    "core_deposit", "core_loan", "core_txn_account",
    "card", "card_txn", "crm_interaction",
]
inventory_sql = " UNION ALL ".join(
    f"SELECT '{table}' AS table_name, count(*) AS row_count, min(cob_dt) AS min_cob_dt, max(cob_dt) AS max_cob_dt FROM lakehouse.bronze.{table}"
    for table in BRONZE_TABLES
)
bronze_inventory = spark.sql(inventory_sql).orderBy("table_name")
bronze_inventory.show(50, truncate=False)
assert bronze_inventory.count() == len(BRONZE_TABLES)

+----------------+---------+----------+----------+
|table_name      |row_count|min_cob_dt|max_cob_dt|
+----------------+---------+----------+----------+
|card            |30000    |2025-12-31|2026-07-19|
|card_txn        |212400   |2025-01-05|2025-12-31|
|core_account    |40500    |2025-12-31|2026-07-19|
|core_branch     |150      |2025-12-31|2026-07-19|
|core_customer   |30000    |2025-12-31|2026-07-19|
|core_deposit    |10500    |2025-12-31|2026-07-19|
|core_loan       |8100     |2025-12-31|2026-07-19|
|core_product    |48       |2025-12-31|2026-07-19|
|core_txn_account|1050000  |2025-01-01|2025-12-31|
|crm_interaction |10000    |2025-01-05|2025-12-31|
+----------------+---------+----------+----------+



## 2. Physical schema contract

Sau migration, cả bốn bảng phải có `effective_from`, `effective_to`, `is_current` và surrogate key tương ứng.

In [3]:
required_metadata = {"effective_from", "effective_to", "is_current"}
schema_ready = {}
for table, (_, sk_column) in SCD2_DIMS.items():
    columns = set(spark.table(f"lakehouse.silver.{table}").columns)
    missing = sorted((required_metadata | {sk_column}) - columns)
    schema_ready[table] = not missing
    print(f"{table}: {'READY' if not missing else 'MISSING ' + str(missing)}")
schema_ready

dim_branch: READY
dim_product: READY
dim_customer: READY
dim_account: READY


{'dim_branch': True,
 'dim_product': True,
 'dim_customer': True,
 'dim_account': True}

Nếu branch/product còn báo `MISSING`, đó là trạng thái trước migration. Chỉ chạy phần baseline tiếp theo sau khi reset riêng Silver/Gold và load Silver `2025-12-31`.

In [4]:
assert all(schema_ready.values()), "Schema SCD2 chưa sẵn sàng; dừng trước baseline acceptance"

baseline_parts = []
for table, (business_key, _) in SCD2_DIMS.items():
    baseline_parts.append(f"""
        SELECT '{table}' AS table_name,
               count(*) AS total_rows,
               sum(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) AS current_rows,
               sum(CASE WHEN is_current = 0 THEN 1 ELSE 0 END) AS history_rows,
               count(DISTINCT {business_key}) AS business_keys,
               min(effective_from) AS min_effective_from,
               max(effective_from) AS max_effective_from
        FROM lakehouse.silver.{table}
    """)
baseline = spark.sql(" UNION ALL ".join(baseline_parts)).orderBy("table_name")
baseline.show(truncate=False)

for row in baseline.collect():
    assert row.total_rows == row.current_rows == row.business_keys, row
    assert row.history_rows == 0, row
    assert str(row.min_effective_from) == BASE_COB_DT, row
    assert str(row.max_effective_from) == BASE_COB_DT, row
print("BASELINE ACCEPTED")

+------------+----------+------------+------------+-------------+------------------+------------------+
|table_name  |total_rows|current_rows|history_rows|business_keys|min_effective_from|max_effective_from|
+------------+----------+------------+------------+-------------+------------------+------------------+
|dim_account |13500     |13500       |0           |13500        |2025-12-31        |2025-12-31        |
|dim_branch  |50        |50          |0           |50           |2025-12-31        |2025-12-31        |
|dim_customer|10000     |10000       |0           |10000        |2025-12-31        |2025-12-31        |
|dim_product |16        |16          |0           |16           |2025-12-31        |2025-12-31        |
+------------+----------+------------+------------+-------------+------------------+------------------+



BASELINE ACCEPTED


## Tiếp theo

1. Chạy `scd2_demo_changes_oracle.sql`.
2. Trigger `bronze_core_banking_dag` và `bronze_card_crm_dag` với `cob_dt=2026-01-01`.
3. Trigger `silver_all_dag` với cùng ngày.
4. Mở notebook `02_scd2_day2_acceptance.ipynb`.